In [2]:
## CREATE MASTER DATASET

import pandas as pd

calendar = pd.read_csv("./data/calendar_afcs2025.csv")
sell = pd.read_csv("./data/sell_prices_afcs2025.csv")
train = pd.read_csv("./data/sales_train_validation_afcs2025.csv")
test_v = pd.read_csv("./data/sales_test_validation_afcs2025.csv")
test_e = pd.read_csv("./data/sales_test_evaluation_afcs_2025.csv")

calendar["d"] = [f"d_{i}" for i in range(1, len(calendar) + 1)]
calendar["date"] = pd.to_datetime(calendar["date"])

def add_keys(df):
    p = df["id"].str.split("_", expand=True)
    df = df.copy()
    df["cat_id"]   = p[0]
    df["dept_id"]  = p[0] + "_" + p[1]
    df["item_id"]  = p[0] + "_" + p[1] + "_" + p[2]
    df["state_id"] = p[3]
    df["store_id"] = p[3] + "_" + p[4]
    df["split"]    = p[5]
    return df

def melt_sales(df, dataset_name):
    df = add_keys(df)
    dcols = [c for c in df.columns if c.startswith("d_")]
    long = df.melt(
        id_vars=["id","cat_id","dept_id","item_id","state_id","store_id","split"],
        value_vars=dcols, var_name="d", value_name="sales"
    )
    long["dataset"] = dataset_name
    return long

master = pd.concat([
    melt_sales(train, "train_validation"),
    melt_sales(test_v, "test_validation"),
    melt_sales(test_e, "test_evaluation"),
], ignore_index=True)

master = master.merge(calendar, on="d", how="left")
master = master.merge(sell, on=["store_id","item_id","wm_yr_wk"], how="left")
master["revenue"] = master["sales"] * master["sell_price"]

master.to_csv("./outputs/master_dataset.csv", index=False)


In [5]:
master

,id,cat_id,dept_id,item_id,state_id,store_id,split,d,sales,dataset,...,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_TX,sell_price,revenue
0,FOODS_3_001_TX_3_validation,FOODS,FOODS_3,FOODS_3_001,TX,TX_3,validation,d_1,0,train_validation,...,1,1,2011,NaN,NaN,NaN,NaN,0,2.28,0.00
1,FOODS_3_002_TX_3_validation,FOODS,FOODS_3,FOODS_3_002,TX,TX_3,validation,d_1,0,train_validation,...,1,1,2011,NaN,NaN,NaN,NaN,0,NaN,NaN
2,FOODS_3_003_TX_3_validation,FOODS,FOODS_3,FOODS_3_003,TX,TX_3,validation,d_1,0,train_validation,...,1,1,2011,NaN,NaN,NaN,NaN,0,NaN,NaN
3,FOODS_3_004_TX_3_validation,FOODS,FOODS_3,FOODS_3_004,TX,TX_3,validation,d_1,0,train_validation,...,1,1,2011,NaN,NaN,NaN,NaN,0,NaN,NaN
4,FOODS_3_005_TX_3_validation,FOODS,FOODS_3,FOODS_3_005,TX,TX_3,validation,d_1,0,train_validation,...,1,1,2011,NaN,NaN,NaN,NaN,0,1.68,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1620482,FOODS_3_823_TX_3_validation,FOODS,FOODS_3,FOODS_3_823,TX,TX_3,validation,d_1969,0,test_evaluation,...,2,6,2016,NBAFinalsEnd,Sporting,Father's day,Cultural,0,2.98,0.00
1620483,FOODS_3_824_TX_3_validation,FOODS,FOODS_3,FOODS_3_824,TX,TX_3,validation,d_1969,0,test_evaluation,...,2,6,2016,NBAFinalsEnd,Sporting,Father's day,Cultural,0,2.48,0.00
1620484,FOODS_3_825_TX_3_validation,FOODS,FOODS_3,FOODS_3_825,TX,TX_3,validation,d_1969,0,test_evaluation,...,2,6,2016,NBAFinalsEnd,Sporting,Father's day,Cultural,0,3.98,0.00
1620485,FOODS_3_826_TX_3_validation,FOODS,FOODS_3,FOODS_3_826,TX,TX_3,validation,d_1969,1,test_evaluation,...,2,6,2016,NBAFinalsEnd,Sporting,Father's day,Cultural,0,1.28,1.28
